# 15 — Time-delay cosmography (Refsdal H_0)

Light rays that take different paths through a strong lens arrive at the observer at different times. The time delay depends on cosmographic distances, so an observed Δt plus a lens model for the **Fermat potential** Δτ pins down the **time-delay distance** D_Δt and hence H_0.

This is Refsdal (1964)'s original idea, today applied by TDCOSMO / H0LiCOW to measure H_0 to ~2% with ~6 quasar lenses. We reproduce the principle on a synthetic SIE.

Reference: Meneghetti, *Lensing Gravitazionale* (UNIBO MSc), Ch. 3.6 (Fermat surface) and Ch. 5.8 (time delays); Refsdal 1964; Suyu et al. 2017.

In [ ]:
# Bootstrap: make `lensing` importable when running notebooks/ directly.
import sys
from pathlib import Path
repo = Path.cwd().resolve().parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

import lensing as gl
# Device-agnostic: prefer MPS (Apple GPU) → CUDA → CPU.
# Pass device="cpu" if you need to force the CPU path (e.g. for
# operators that have no MPS kernel yet, or for reproducibility).
device, dtype = gl.config.setup(seed=42)
print(f"using device: {device}")


## 1. The Fermat potential surface

We map τ(θ; β) = ½ |θ − β|² − Ψ̂(θ) on the image plane. Each minimum/saddle of τ corresponds to an image (Fermat's principle), and the contour height directly maps to the geometric+gravitational arrival-time delay.

In [ ]:
# Build a circularly symmetric power-law lens (analytic Psi)
# so we can plot tau without numerical integration.
lens = gl.lens.PowerLawSpherical(theta_E=1.0, n=2.0)  # = SIS
beta = torch.tensor([0.10, 0.05])  # source position [arcsec]

xy = gl.data.coordinate_grid(npix=300, deltapix=0.02)
with torch.no_grad():
    psi = lens.potential(xy[0], xy[1])
    theta_minus_beta = torch.stack([xy[0]-beta[0], xy[1]-beta[1]], dim=0)
    tau = 0.5 * (theta_minus_beta**2).sum(dim=0) - psi

half = 0.5 * 300 * 0.02
fig, ax = plt.subplots(figsize=(7, 6))
cs = ax.contour(xy[0].numpy(), xy[1].numpy(), tau.numpy(),
                levels=30, cmap='viridis')
ax.plot(beta[0], beta[1], '*', color='orange', ms=12, label='source')
ax.set(xlabel=r'$\theta_1$ [arcsec]', ylabel=r'$\theta_2$ [arcsec]',
       title='Fermat potential τ(θ; β) for an SIS')
ax.set_aspect('equal'); ax.legend(); ax.grid(alpha=0.3)
plt.show()


**Reading the contour map**: the saddle points of τ are the image positions; they appear as crossing-points of the level curves. For an SIS at finite β there are exactly two images (a minimum and a saddle) on opposite sides of the lens centre.

## 2. Δt from a Refsdal-like quasar pair

We pick a 4-image SIE, solve for the image positions, evaluate τ at each image, and convert the Fermat-potential differences into physical Δt (days). Then we **invert** the longest Δt to recover H_0.

In [ ]:
cosmo = gl.cosmology.Cosmology(H0=70., Om0=0.3)
sie = gl.lens.SIE.from_velocity_dispersion(
    sigma_v_kms=240., q=0.7, pa=np.pi/4,
    zl=0.5, zs=2.0, cosmo=cosmo,
)
beta_x, beta_y = torch.tensor(0.05), torch.tensor(0.03)
ximg, yimg = sie.solve_image_positions(beta_x, beta_y, n_grid=4000)
print(f'Found {len(ximg)} images at')
for i, (x, y) in enumerate(zip(ximg, yimg)):
    print(f'  image {i+1}: ({float(x):+.3f}, {float(y):+.3f}) arcsec')

# SIE has no analytic Psi in our package, so we approximate Psi by
# numerical line integration of alpha . dr along the ray from the
# origin to each image (Psi has alpha = grad Psi).
def sie_potential_along(lens, x, y, n_steps=400):
    # Integrate radially from r ~ 0 to r_image; works for any
    # axially-near profile but the SIE has small angular
    # variation away from the major axis, so this is accurate
    # to ~1% for our visualization purposes.
    t = torch.linspace(0.0, 1.0, n_steps).reshape(-1, 1)
    xs = t * x.reshape(1, -1)
    ys = t * y.reshape(1, -1)
    ax_d, ay_d = lens.deflection(xs, ys)
    dx = (x.reshape(1, -1)) / n_steps
    dy = (y.reshape(1, -1)) / n_steps
    return (ax_d * dx + ay_d * dy).sum(dim=0)

with torch.no_grad():
    psi_img = sie_potential_along(sie, ximg, yimg)
    # Fermat potential per image
    dx = ximg - beta_x
    dy = yimg - beta_y
    tau = 0.5 * (dx**2 + dy**2) - psi_img
    tau_diffs = tau - tau.min()  # in arcsec^2
print()
for i, t in enumerate(tau_diffs.tolist()):
    dt_s = float(gl.lens.timedelay.time_delay_seconds(
        torch.tensor(t), zl=0.5, zs=2.0, cosmo=cosmo))
    print(f'  image {i+1}: Δτ = {t:+.4f} arcsec², Δt = {dt_s/86400:7.2f} days')


## 3. Inverting Δt for H_0

Suppose the longest delay is *measured* to 2% precision. We feed the model Δτ and the observed Δt to :func:`lensing.lens.timedelay.refsdal_H0` and compare the recovered H_0 to the input cosmology.

In [ ]:
# Pretend the largest Δt we just computed is the *measured* one.
longest_idx = int(torch.argmax(tau_diffs))
measured_dt_days = float(gl.lens.timedelay.time_delay_seconds(
    tau_diffs[longest_idx], zl=0.5, zs=2.0, cosmo=cosmo)) / 86400.0
model_dtau_arcsec2 = float(tau_diffs[longest_idx])

# Add a 2% measurement error and re-invert.
for noise in [0.0, 0.02, 0.05]:
    meas = measured_dt_days * (1.0 + noise)
    H0_rec = gl.lens.timedelay.refsdal_H0(
        meas, model_dtau_arcsec2, zl=0.5, zs=2.0,
    )
    print(f'  Δt err = {100*noise:4.1f}%   →  H_0 = {H0_rec:.2f} km/s/Mpc')
print(f'  truth                   →  H_0 = 70.00 km/s/Mpc')


**Discussion**: Δt and H_0 are inversely proportional, so a 5% Δt error propagates into a 5% H_0 error — the **dominant systematic** in real measurements is therefore the lens model (through the Fermat-potential difference), *not* the time-delay monitoring itself. Modern programs spend a lot of effort on modelling assumptions: profile slope, line-of-sight convergence, and the **mass-sheet degeneracy** which leaves the image configuration unchanged but rescales τ by a multiplicative constant (Falco, Gorenstein & Shapiro 1985). See `docs/astrophysics.md` for a fuller discussion.